# 3. Train
Trains the seq2seq (BiLSTM encoder + attention decoder) sign-language translator on the quarter-subset, using the features from notebook 2.

Saves `models/best_sign_language_model.pt` and `models/vocab.pkl`.

In [ ]:
!pip install -q torch pandas numpy scikit-learn

In [ ]:
import os
import pickle
import random
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Vocabulary

In [ ]:
class Vocabulary:
    def __init__(self, freq_threshold=1):
        self.word2idx = {"<pad>": 0, "<sos>": 1, "<eos>": 2, "<unk>": 3}
        self.idx2word = {v: k for k, v in self.word2idx.items()}
        self.freq_threshold = freq_threshold
        self.word_counts = Counter()

    def build_vocabulary(self, sentence_list):
        for sent in sentence_list:
            self.word_counts.update(sent.lower().split())
        for word, count in self.word_counts.items():
            if count >= self.freq_threshold:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx] = word

    def numericalize(self, text):
        tokens = text.lower().split()
        return [self.word2idx.get(t, self.word2idx["<unk>"]) for t in tokens]

    def __len__(self):
        return len(self.word2idx)

## Dataset & collate function

In [ ]:
class SignLanguageDataset(Dataset):
    def __init__(self, df, features_dir, vocab):
        self.df = df.reset_index(drop=True)
        self.features_dir = features_dir
        self.vocab = vocab

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        video_id = str(row["video_id"])
        text = str(row["gloss"])

        feat_path = os.path.join(self.features_dir, f"{video_id}.npy")
        features = np.load(feat_path).astype(np.float32)

        tokens = [self.vocab.word2idx["<sos>"]]
        tokens += self.vocab.numericalize(text)
        tokens.append(self.vocab.word2idx["<eos>"])

        return torch.FloatTensor(features), torch.LongTensor(tokens)


def collate_fn(batch):
    features, targets = zip(*batch)

    feat_lengths = [f.shape[0] for f in features]
    max_feat_len = max(feat_lengths)
    feat_dim = features[0].shape[1]
    padded_feats = torch.zeros(len(features), max_feat_len, feat_dim)
    for i, f in enumerate(features):
        padded_feats[i, :f.shape[0]] = f

    tgt_lengths = [t.shape[0] for t in targets]
    max_tgt_len = max(tgt_lengths)
    padded_tgts = torch.full((len(targets), max_tgt_len), fill_value=0)
    for i, t in enumerate(targets):
        padded_tgts[i, :t.shape[0]] = t

    return (padded_feats, padded_tgts,
            torch.LongTensor(feat_lengths), torch.LongTensor(tgt_lengths))

## Model: Encoder, Attention, Decoder, Seq2Seq wrapper

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim=1662, hid_dim=512, n_layers=2, dropout=0.3):
        super().__init__()
        self.hid_dim = hid_dim
        self.n_layers = n_layers

        self.input_proj = nn.Linear(input_dim, hid_dim)
        self.lstm = nn.LSTM(hid_dim, hid_dim, n_layers,
                             batch_first=True, bidirectional=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

        self.fc_h = nn.Linear(hid_dim * 2, hid_dim)
        self.fc_c = nn.Linear(hid_dim * 2, hid_dim)

    def forward(self, x, lengths):
        x = self.dropout(self.input_proj(x))

        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_outputs, (hidden, cell) = self.lstm(packed)
        outputs, _ = pad_packed_sequence(packed_outputs, batch_first=True)

        hidden = hidden.view(self.n_layers, 2, hidden.size(1), hidden.size(2))
        hidden = torch.cat((hidden[:, 0], hidden[:, 1]), dim=2)
        hidden = self.fc_h(hidden)

        cell = cell.view(self.n_layers, 2, cell.size(1), cell.size(2))
        cell = torch.cat((cell[:, 0], cell[:, 1]), dim=2)
        cell = self.fc_c(cell)

        return outputs, hidden, cell


class Attention(nn.Module):
    def __init__(self, enc_hid_dim=512, dec_hid_dim=512):
        super().__init__()
        self.attn = nn.Linear((enc_hid_dim * 2) + dec_hid_dim, dec_hid_dim)
        self.v = nn.Linear(dec_hid_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        batch_size, src_len, _ = encoder_outputs.shape

        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)

        weights = torch.softmax(attention, dim=1)
        context = torch.bmm(weights.unsqueeze(1), encoder_outputs).squeeze(1)

        return context, weights


class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim=256, enc_hid_dim=512,
                 dec_hid_dim=512, n_layers=2, dropout=0.3):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.attention = Attention(enc_hid_dim, dec_hid_dim)

        self.lstm = nn.LSTM(emb_dim + (enc_hid_dim * 2), dec_hid_dim,
                             n_layers, batch_first=True, dropout=dropout)
        self.fc_out = nn.Linear(dec_hid_dim + (enc_hid_dim * 2) + emb_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell, encoder_outputs):
        input = input.unsqueeze(1)
        embedded = self.dropout(self.embedding(input))

        context, attn_w = self.attention(hidden[-1], encoder_outputs)
        context = context.unsqueeze(1)

        lstm_input = torch.cat((embedded, context), dim=2)
        output, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))

        embedded = embedded.squeeze(1)
        output = output.squeeze(1)
        context = context.squeeze(1)

        prediction = self.fc_out(torch.cat((output, context, embedded), dim=1))

        return prediction, hidden, cell, attn_w


class SignLanguageTranslator(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, src_lengths, trg, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)

        encoder_outputs, hidden, cell = self.encoder(src, src_lengths)

        input = trg[:, 0]

        for t in range(1, trg_len):
            output, hidden, cell, _ = self.decoder(input, hidden, cell, encoder_outputs)
            outputs[:, t] = output

            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[:, t] if teacher_force else top1

        return outputs

## Training & evaluation loops

In [ ]:
def train_epoch(model, iterator, optimizer, criterion, clip):
    model.train()
    epoch_loss = 0
    for batch in iterator:
        src, trg, src_len, trg_len = [x.to(device) for x in batch]

        optimizer.zero_grad()
        output = model(src, src_len, trg, teacher_forcing_ratio=0.5)

        output_dim = output.shape[-1]
        output = output[:, 1:].reshape(-1, output_dim)
        trg = trg[:, 1:].reshape(-1)

        loss = criterion(output, trg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

        epoch_loss += loss.item()
    return epoch_loss / len(iterator)


def evaluate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for batch in iterator:
            src, trg, src_len, trg_len = [x.to(device) for x in batch]
            output = model(src, src_len, trg, teacher_forcing_ratio=0.0)

            output_dim = output.shape[-1]
            output = output[:, 1:].reshape(-1, output_dim)
            trg = trg[:, 1:].reshape(-1)

            loss = criterion(output, trg)
            epoch_loss += loss.item()
    return epoch_loss / len(iterator)


def translate(model, features, vocab, device, max_len=50):
    model.eval()

    if isinstance(features, np.ndarray):
        features = torch.FloatTensor(features)

    features = features.unsqueeze(0).to(device)
    src_len = torch.LongTensor([features.shape[1]]).to(device)

    with torch.no_grad():
        encoder_outputs, hidden, cell = model.encoder(features, src_len)

    inputs = torch.LongTensor([vocab.word2idx["<sos>"]]).to(device)
    outputs = []

    for _ in range(max_len):
        with torch.no_grad():
            output, hidden, cell, _ = model.decoder(inputs, hidden, cell, encoder_outputs)

        pred = output.argmax(1).item()

        if pred == vocab.word2idx["<eos>"]:
            break

        outputs.append(vocab.idx2word[pred])
        inputs = torch.LongTensor([pred]).to(device)

    return " ".join(outputs)

## Load data, build vocab

In [ ]:
CSV_PATH = "wlasl_quarter.csv"
FEATURES_DIR = "extracted_features"
MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

df = pd.read_csv(CSV_PATH)
df["has_features"] = df["video_id"].apply(
    lambda vid: os.path.exists(os.path.join(FEATURES_DIR, f"{vid}.npy"))
)
df_ready = df[df["has_features"]].copy().drop(columns=["has_features"])
print(f"Total videos in CSV: {len(df)} | with extracted features: {len(df_ready)}")

if len(df_ready) == 0:
    raise ValueError(
        f"No .npy files found in '{FEATURES_DIR}'. Run notebook 02_extract_features first!"
    )

all_sentences = df_ready["gloss"].astype(str).tolist()
vocab = Vocabulary(freq_threshold=1)
vocab.build_vocabulary(all_sentences)
print(f"Vocabulary size: {len(vocab)}")

with open(os.path.join(MODEL_DIR, "vocab.pkl"), "wb") as f:
    pickle.dump(vocab, f)

## Train/val split + data loaders

In [ ]:
BATCH_SIZE = 16

train_df = df_ready.sample(frac=0.9, random_state=SEED)
val_df = df_ready.drop(train_df.index)

train_loader = DataLoader(
    SignLanguageDataset(train_df, FEATURES_DIR, vocab),
    batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(
    SignLanguageDataset(val_df, FEATURES_DIR, vocab),
    batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Train: {len(train_df)} | Val: {len(val_df)}")

## Build model

In [ ]:
HID_DIM = 512
ENC_LAYERS = 2
DEC_LAYERS = 2
ENC_DROPOUT = 0.3
DEC_DROPOUT = 0.3
LEARNING_RATE = 0.001
CLIP = 1.0

enc = Encoder(input_dim=1662, hid_dim=HID_DIM, n_layers=ENC_LAYERS, dropout=ENC_DROPOUT)
dec = Decoder(output_dim=len(vocab), emb_dim=256, enc_hid_dim=HID_DIM,
              dec_hid_dim=HID_DIM, n_layers=DEC_LAYERS, dropout=DEC_DROPOUT)
model = SignLanguageTranslator(enc, dec, device).to(device)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss(ignore_index=0)

print(model)

## Train

In [ ]:
N_EPOCHS = 50
CKPT_PATH = os.path.join(MODEL_DIR, "best_sign_language_model.pt")
best_val_loss = float("inf")

for epoch in range(N_EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, CLIP)
    val_loss = evaluate(model, val_loader, criterion)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), CKPT_PATH)
        print("  -> Saved new best model")

    print(f"Epoch {epoch+1:02d}: Train Loss = {train_loss:.4f} | Val Loss = {val_loss:.4f}")

## Sample translation from the best checkpoint

In [ ]:
sample_video_id = val_df.iloc[0]["video_id"]
sample_text = val_df.iloc[0]["gloss"]
sample_features = np.load(os.path.join(FEATURES_DIR, f"{sample_video_id}.npy"))

model.load_state_dict(torch.load(CKPT_PATH))
translation = translate(model, sample_features, vocab, device)

print(f"Video ID : {sample_video_id}")
print(f"Target   : {sample_text}")
print(f"Predicted: {translation}")

## (Colab only) Copy trained model to Google Drive so it survives a runtime reset

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copytree(MODEL_DIR, '/content/drive/MyDrive/wlasl_models', dirs_exist_ok=True)
# print("Copied to Drive")